# RAG audit demonstration

Connect FAISS retrieval to a structured NASA requirement audit. The saved outputs come from the local model run documented below. Set `RUN_MODEL = True` to rerun inference. Schema and citation validity do not establish verdict accuracy.

In Colab, choose a GPU runtime, then run all cells. Setup downloads the public repository and installs dependencies. Locally, open this notebook from the checkout after installing `requirements.txt`.

In [1]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/Aerospace-1C-compliance-copilot")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--branch", "alan/rag-audit-pipeline",
                        "https://github.com/Break-Through-Tech/Aerospace-1C-compliance-copilot.git",
                        str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "compliance_copilot").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository ready")

Repository ready


In [2]:
RUN_MODEL = False
REPORT = ROOT / "outputs/rag/demo.json"
if RUN_MODEL:
    REPORT = ROOT / "outputs/rag/notebook-run.json"
    REPORT.unlink(missing_ok=True)
    completed = subprocess.run([sys.executable, str(ROOT / "scripts/run_rag_demo.py"),
                                "--output", str(REPORT)], cwd=ROOT)
    if not REPORT.exists():
        raise RuntimeError("Model run failed before writing results; inspect the output above")
    if completed.returncode:
        print("The run contains audit errors; inspect them below.")
print("Displaying a new run" if RUN_MODEL else "Displaying the saved model run")

Displaying the saved model run


In [3]:
import json
import pandas as pd
from compliance_copilot.schemas import AuditResult

report = json.loads(REPORT.read_text())
results = [AuditResult.model_validate(row) for row in report["results"]]
print(json.dumps(report["config"], indent=2))
pd.DataFrame([{"requirement_id": r.requirement_id, "status": r.status,
               "verdict": r.audit.verdict if r.audit else None,
               "cited_swe_ids": ", ".join(c.swe_id for c in r.audit.citations) if r.audit else "",
               "error": r.error_code} for r in results])

{
  "backend": "transformers",
  "model": "Qwen/Qwen3-4B-Instruct-2507",
  "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "top_k": 10,
  "normalize_agency": true,
  "do_sample": false,
  "max_new_tokens": 768,
  "device": "mps",
  "python": "3.12.13",
  "packages": {
    "torch": "2.14.1",
    "transformers": "4.57.6",
    "sentence-transformers": "5.7.0",
    "faiss-cpu": "1.15.1",
    "pydantic": "2.13.5"
  }
}


,requirement_id,status,verdict,cited_swe_ids,error
0,NASA-SR-001,ok,Meets,"SWE-013, SWE-125",None
1,NASA-SR-002,ok,Gap,"SWE-199, SWE-055",None
2,NASA-SR-003,ok,Meets,"SWE-052, SWE-053",None
3,NASA-SR-004,ok,Meets,SWE-050,None
4,NASA-SR-005,ok,Gap,SWE-053,None
5,NASA-SR-006,ok,Meets,SWE-055,None
6,NASA-SR-007,ok,Meets,"SWE-057, SWE-081, SWE-083",None
7,NASA-SR-008,ok,Partial,SWE-061,None
8,NASA-SR-009,ok,Meets,"SWE-135, SWE-201",None
9,NASA-SR-010,ok,Meets,"SWE-062, SWE-068",None


In [4]:
for result in results:
    print(result.requirement_id)
    print(result.requirement_text)
    if result.audit:
        print(result.audit.verdict + ": " + result.audit.reasoning)
        for citation in result.audit.citations:
            clause = next(c for c in result.retrieved if c.swe_id == citation.swe_id)
            print(citation.swe_id, citation.section, repr(citation.quote))
            print(clause.source_url)
    else:
        print(result.error_code, result.error_message)
    print()

NASA-SR-001
The NASA Project Manager shall develop, maintain, and execute a Software Development Plan and a Software Security Plan covering the full software life cycle from requirements through retirement, addressing every applicable requirement of NPR 7150.2D per the approved Requirements Mapping Matrix.
Meets: SWE-013 mandates development and execution of software plans covering the full life cycle and addressing NPR 7150.2D requirements with approved tailoring. SWE-125 requires a requirements mapping matrix to align with NPR requirements. The proposed requirement satisfies both by explicitly stating development, maintenance, execution of software and security plans, and reference to the approved mapping matrix.
SWE-013 3.1.3 '3.1.3 The project manager shall develop, maintain, and execute software plans, including security plans, that cover the entire software life cycle and, as a minimum, address the requirements of this directive with approved tailoring. [SWE-013]'
https://explore

In [5]:
# Labels are joined only after inference, never supplied to the pipeline.
benchmark = pd.read_csv(ROOT / "data/benchmark/requirements.csv").set_index("requirement_id")
pd.DataFrame([{"requirement_id": r.requirement_id,
               "expected": benchmark.loc[r.requirement_id, "verdict"],
               "predicted": r.audit.verdict if r.audit else None,
               "expected_clause_retrieved": benchmark.loc[r.requirement_id, "swe_id"] in {c.swe_id for c in r.retrieved}}
              for r in results])

,requirement_id,expected,predicted,expected_clause_retrieved
0,NASA-SR-001,Meets,Meets,True
1,NASA-SR-002,Gap,Gap,True
2,NASA-SR-003,Meets,Meets,True
3,NASA-SR-004,Meets,Meets,True
4,NASA-SR-005,Partial,Gap,True
5,NASA-SR-006,Meets,Meets,True
6,NASA-SR-007,Meets,Meets,True
7,NASA-SR-008,Gap,Partial,True
8,NASA-SR-009,Meets,Meets,True
9,NASA-SR-010,Meets,Meets,True


In [6]:
from scripts.summarize_rag_run import summarize

cases = pd.read_csv(ROOT / "data/benchmark/requirements.csv").to_dict(orient="records")
comparisons = []
for label, filename in [("Original 10", "baseline.json"),
                        ("Updated 10", "demo.json"),
                        ("Full 50", "audit-results.json")]:
    saved = json.loads((ROOT / "outputs/rag" / filename).read_text())
    summary = summarize(saved, cases)
    comparisons.append({"run": label, "total": summary["total"], **summary["counts"]})
pd.DataFrame(comparisons)

,run,total,valid_output,expected_clause_retrieved,expected_clause_cited,verdict_matches,grounded_verdict_matches
0,Original 10,10,7,7,6,5,4
1,Updated 10,10,10,10,9,8,7
2,Full 50,50,50,50,48,37,35


## Interpretation

These examples were used during development and error analysis. They are not an independent holdout. The notebook retains the original ten-example run alongside the updated ten-example slice and the full 50-case run.

A grounded verdict match requires both the expected verdict and the assigned SWE ID in the citations. Correct labels on different clauses do not count as grounded matches. Failures remain in the denominator.

See `docs/rag-quality-check.md` for the measured comparison and remaining cases, and `docs/rag-pipeline.md` for the callable interface, source limitations, and reproduction settings. The team can use these saved outputs in its broader evaluation.